<a href="https://colab.research.google.com/github/moeeed2006-ops/Abdul-Moeed-flyrank-ml-work/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moeeed2006-ops/Abdul-Moeed-flyrank-ml-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row = One unique URL (Individual Page).
**Time Window:** Mid-panel month `2026-03` (March 2026) for feature engineering and label verification, treating `2026-06` as a sealed test month.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
from google.colab import userdata

# 1. Retrieve token securely from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Connect DuckDB and register Hugging Face secret
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# Path to the month=2026-03 partition under fact_content_daily_performance
path_03 = "'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'"

# Query 1: Prove Grain
q1 = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT content_hash_id) as unique_pages
FROM {path_03}
"""
print("--- Query 1: Grain Verification ---")
print(con.execute(q1).df())

# Query 2: Prove Slice Row Count & Date Span
q2 = f"""
SELECT
    COUNT(*) as row_count,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date
FROM {path_03}
"""
print("\n--- Query 2: Row Count & Date Span ---")
print(con.execute(q2).df())

# Query 3: Prove Availability (Filtering with IS TRUE)
q3 = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT_IF(gsc_data_available IS TRUE) as available_rows
FROM {path_03}
"""
print("\n--- Query 3: Availability Filtering ---")
print(con.execute(q3).df())

--- Query 1: Grain Verification ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_pages
0     9841378        331437

--- Query 2: Row Count & Date Span ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31

--- Query 3: Availability Filtering ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  available_rows
0     9841378       3611061.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Features (5 max, each with knowability reason):**
  1. `gsc_clicks` — Knowable at decision moment because past click performance is logged continuously in daily Search Console reports.
  2. `gsc_impressions` — Knowable at decision moment because search visibility counts are captured historically before prediction.
  3. `ga4_sessions` — Knowable at decision moment because user traffic sessions are recorded directly by analytics prior to the evaluation window.
  4. `ga4_engaged_sessions` — Knowable at decision moment because engagement metrics are derived from completed user sessions.
  5. `is_high_impression` — Knowable at decision moment because it is a simple threshold boolean (`gsc_impressions > 100`) calculated from historical visibility logs.

* **Label:** `is_decayed` — Binary target representing severe traffic drop in the observation window.
* **Context:** `content_hash_id` — Entity identifier for unique pages.
* **Excluded (The Trap / Leakage):** `future_click_trend` — Deliberately excluded because it uses clicks from the target evaluation period itself, causing massive data leakage.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# 1. Fetch data: Honest features explicitly DO NOT include the direct target components
query_features = f"""
SELECT
    content_hash_id,
    -- 5 Clean Features (Historical / Raw proxies)
    gsc_clicks,
    gsc_impressions,
    (gsc_clicks / NULLIF(gsc_impressions, 0)) AS calculated_ctr,
    (gsc_impressions > 100)::INT AS is_high_impression,
    (gsc_clicks > 50)::INT AS is_high_traffic,

    -- Target Label Proxy: Decay defined on traffic/engagement drop
    ((gsc_clicks > 20) AND (ga4_engaged_sessions = 0))::INT AS is_decayed,

    -- THE TRAP: Direct Leakage Column (Calculated straight from future target variable outcome)
    (ga4_engaged_sessions = 0)::INT AS LEAKED_zero_engagement_flag

FROM {path_03}
WHERE gsc_data_available IS TRUE
LIMIT 50000
"""

df_feat = con.execute(query_features).df().fillna(0)

# Honest Feature Set (Excludes ga4_engaged_sessions and the leaked flag)
honest_cols = ['gsc_clicks', 'gsc_impressions', 'calculated_ctr', 'is_high_impression', 'is_high_traffic']
X_honest = df_feat[honest_cols]
y = df_feat['is_decayed']

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)

# 2. Honest Model
clf_honest = DecisionTreeClassifier(max_depth=2, random_state=42)
clf_honest.fit(X_train, y_train)
honest_acc = accuracy_score(y_test, clf_honest.predict(X_test))

# 3. Leaked Model (Springing the Trap)
X_leaked = X_honest.copy()
X_leaked['LEAKED_zero_engagement_flag'] = df_feat['LEAKED_zero_engagement_flag']

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaked, y, test_size=0.2, random_state=42)
clf_leaked = DecisionTreeClassifier(max_depth=2, random_state=42)
clf_leaked.fit(X_tr_l, y_tr_l)
leaked_acc = accuracy_score(y_te_l, clf_leaked.predict(X_te_l))

print("--- DATA LEAKAGE EXPERIMENT RESULTS ---")
print(f"Honest Model Score (No Leak):   {honest_acc:.4f}")
print(f"Leaked Model Score (The Trap):  {leaked_acc:.4f}")

# Deleting the trap feature
del df_feat['LEAKED_zero_engagement_flag']
print("\n[CLEANUP] Leaked column 'LEAKED_zero_engagement_flag' permanently deleted.")

--- DATA LEAKAGE EXPERIMENT RESULTS ---
Honest Model Score (No Leak):   0.9999
Leaked Model Score (The Trap):  0.9999

[CLEANUP] Leaked column 'LEAKED_zero_engagement_flag' permanently deleted.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Fact 1: Verify Grain (Unique Pages vs Total Rows)
q_grain = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT content_hash_id) as unique_pages
FROM {path_03}
"""
print("--- Fact 1: Grain Verification ---")
print(con.execute(q_grain).df())

# Fact 2: Verify Window & Slice Row Count
q_window = f"""
SELECT
    COUNT(*) as row_count,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM {path_03}
"""
print("\n--- Fact 2: Window & Row Count ---")
print(con.execute(q_window).df())

# Fact 3: Verify Missing Values / Availability Filtering
q_avail = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT_IF(gsc_data_available IS TRUE) as gsc_available,
    COUNT_IF(ga4_sessions IS NULL) as missing_ga4_rows
FROM {path_03}
"""
print("\n--- Fact 3: Availability & Null Checks ---")
print(con.execute(q_avail).df())

--- Fact 1: Grain Verification ---
   total_rows  unique_pages
0     9841378        331437

--- Fact 2: Window & Row Count ---
   row_count start_date   end_date
0    9841378 2026-03-01 2026-03-31

--- Fact 3: Availability & Null Checks ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available  missing_ga4_rows
0     9841378      3611061.0         3018741.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named Data Limitation:** **Survivorship & Seasonality Bias.** Evaluating a single month slice (`2026-03`) fails to capture multi-month search trend fluctuations or annual seasonality. Furthermore, pages deleted or redirected prior to March 2026 are completely absent from this warehouse snapshot, introducing survivorship bias.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify tracking limits: GSC-only rows and missing GA4 matches
q_limit = f"""
SELECT
    COUNT_IF(gsc_data_available IS TRUE AND (ga4_sessions IS NULL OR ga4_sessions = 0)) as gsc_only_unmatched_rows,
    COUNT_IF(gsc_clicks > 0 AND (ga4_sessions IS NULL OR ga4_sessions = 0)) as tracking_discrepancy_rows
FROM {path_03}
"""
print("--- Data Limits Verification ---")
print(con.execute(q_limit).df())

--- Data Limits Verification ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   gsc_only_unmatched_rows  tracking_discrepancy_rows
0                3249966.0                   237870.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.